In [ ]:
# ╔═══════════════════════════════════════════════════════════════════════════╗
# ║                                                                         ║
# ║   SECTION A: SHARED PIPELINE (ALL THREE STUDENTS)                       ║
# ║   Cells A1 – A14                                                        ║
# ║                                                                         ║
# ║   Responsible: All three students collaboratively                       ║
# ║   Contents: Installation, imports, data loading, cleaning,              ║
# ║             fire feature computation, feature engineering,              ║
# ║             combine stations, temporal split, EDA                       ║
# ║                                                                         ║
# ╚═══════════════════════════════════════════════════════════════════════════╝

In [ ]:
# Run once in Colab to install all required packages
!pip install keras-tcn shap lightgbm xgboost openpyxl pyarrow -q

All libraries needed across the entire project.
Shared infrastructure used by all three students

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os
import time
from itertools import combinations

from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    precision_score, recall_score, f1_score, roc_auc_score,
    brier_score_loss, classification_report, confusion_matrix
)
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.inspection import PartialDependenceDisplay
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from xgboost import XGBRegressor, XGBClassifier
from lightgbm import LGBMRegressor, LGBMClassifier
import shap

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60)

print("✅ All imports successful.")

Load three source files:
  1. Stations.csv — coordinates of 6 NSW monitoring stations
  2. AIRQuality_Clean.xlsx — hourly PM2.5, temperature, humidity, wind, rainfall
  3. HOTSPOT_Clean.xlsx — satellite fire detections (4 sheets, 2016–2025)

WHY: Air quality provides the target variable (PM2.5) and meteorological
covariates. Hotspots provide wildfire activity data. Station coordinates
are needed to compute distances between fires and monitoring sites.


In [ ]:
# ---- Air Quality ----
df_aq = pd.read_excel("AIRQuality_Clean.xlsx")
df_aq.columns = df_aq.columns.str.strip()

# Rename to consistent snake_case
rename_map = {
    "Source.Name": "station",
    "Wind_Direction": "wind_direction",
    "TEMP": "temp",
    "Wind_speed": "wind_speed",
    "Wind_Speed": "wind_speed",
    "PM25": "pm25",
    "HUMIDITY": "humidity",
    "RAINFALL": "rainfall"
}
rename_map = {k: v for k, v in rename_map.items() if k in df_aq.columns}
df_aq = df_aq.rename(columns=rename_map)

print(f"\nAir Quality shape: {df_aq.shape}")
print(f"Columns: {df_aq.columns.tolist()}")
print(f"Stations in AQ: {sorted(df_aq['station'].unique())}")

Hotspot data spans 4 Excel sheets covering 2016–2025.
We concatenate all into one unified DataFrame.

WHY: Fire detections are split by time period in separate sheets.
We need a single consolidated dataset for efficient lookups when
computing fire-exposure features for each station-hour.


In [ ]:
hotspot_sheets = [
    "Hotspot_2016-2018",
    "Hotspot_2019-2020",
    "Hotspot_2021-2022",
    "Hotspot_2023-2025"
]

hotspot_frames = []
for sheet in hotspot_sheets:
    try:
        df_sh = pd.read_excel("HOTSPOT_Clean.xlsx", sheet_name=sheet)
        df_sh.columns = df_sh.columns.str.strip()
        hotspot_frames.append(df_sh)
        print(f"  Loaded {sheet}: {len(df_sh):,} rows")
    except Exception as e:
        print(f"  Warning: Could not load {sheet}: {e}")

df_hotspots = pd.concat(hotspot_frames, ignore_index=True)

# Standardise column names
hs_rename = {"Source.Name": "source_name"}
hs_rename = {k: v for k, v in hs_rename.items() if k in df_hotspots.columns}
df_hotspots = df_hotspots.rename(columns=hs_rename)

print(f"\nTotal Hotspots: {len(df_hotspots):,}")
print(f"Columns: {df_hotspots.columns.tolist()}")
print(f"Date range: {df_hotspots['datetime'].min()} → {df_hotspots['datetime'].max()}")


Fix the "24:00" hour format and create proper datetime column.

WHY: Raw air quality data uses "24:00" as the last hour of the day,
but this is invalid for datetime parsing. We convert it to "00:00"
of the NEXT day. Correct datetime is essential for all temporal
operations: lags, rolling windows, merges with hourly hotspot data.


In [ ]:
df_aq["station"] = df_aq["station"].str.strip().str.title()

# Drop rows where Date or Time is missing (prevents parsing errors)
df_aq = df_aq.dropna(subset=["Date", "Time"])

# Fix '24:00' / '24:00:00' format
time_str = df_aq["Time"].astype(str).str.strip()
is_24_hr = time_str.str.startswith("24:")
cleaned_time = time_str.copy()
cleaned_time[is_24_hr] = cleaned_time[is_24_hr].str.replace(r"^24:", "00:", regex=True)

# Combine Date + Time into datetime
df_aq["datetime"] = pd.to_datetime(
    df_aq["Date"].astype(str) + " " + cleaned_time,
    format="mixed", dayfirst=True
)

# Add 1 day to rows that originally had hour "24:"
df_aq.loc[is_24_hr, "datetime"] += pd.Timedelta(days=1)

# Sort chronologically per station
df_aq = df_aq.sort_values(["station", "datetime"]).reset_index(drop=True)

# Drop original Date and Time columns (we now have datetime)
df_aq = df_aq.drop(columns=["Date", "Time"], errors="ignore")

print(f"Air Quality after datetime fix: {df_aq.shape}")
print(f"\nPer-station summary:")
for stn in sorted(df_aq["station"].unique()):
    s = df_aq[df_aq["station"] == stn]
    pm_valid = s["pm25"].notna().sum()
    temp_valid = s["temp"].notna().sum()
    print(f"  {stn:15s}: {len(s):6d} rows | PM2.5 valid: {pm_valid:6d} | "
          f"Temp valid: {temp_valid:6d}")


Check data completeness for each station. Stations with >50% missing
PM2.5 (target) or >50% missing temperature (key covariate) are excluded.

WHY: Training models on stations with no target data is impossible.
We document exclusions transparently rather than silently dropping data.


In [ ]:
station_quality = []
for stn in sorted(df_aq["station"].unique()):
    s = df_aq[df_aq["station"] == stn]
    n = len(s)
    station_quality.append({
        "Station": stn,
        "Total Rows": n,
        "PM2.5 Missing %": round(s["pm25"].isna().sum() / n * 100, 1),
        "Temp Missing %": round(s["temp"].isna().sum() / n * 100, 1),
        "Humidity Missing %": round(s["humidity"].isna().sum() / n * 100, 1),
    })

df_quality = pd.DataFrame(station_quality)
print("Station Data Quality:")
print(df_quality.to_string(index=False))

# Determine usable stations
USABLE_STATIONS = df_quality[
    (df_quality["PM2.5 Missing %"] < 50) &
    (df_quality["Temp Missing %"] < 50)
]["Station"].tolist()

EXCLUDED_STATIONS = [s for s in df_quality["Station"] if s not in USABLE_STATIONS]

print(f"\n✅ Usable stations ({len(USABLE_STATIONS)}): {USABLE_STATIONS}")
if EXCLUDED_STATIONS:
    print(f"❌ Excluded ({len(EXCLUDED_STATIONS)}): {EXCLUDED_STATIONS}")
    print("   Reason: >50% missing PM2.5 or temperature")
else:
    print("❌ Excluded: None — all stations have sufficient data")

# Filter to usable stations only
df_aq = df_aq[df_aq["station"].isin(USABLE_STATIONS)].reset_index(drop=True)
print(f"\nFiltered AQ dataset: {df_aq.shape}")


Scientifically correct imputation — different strategies per variable:

PM2.5 (target):
  - Interpolate gaps ≤3 hours only
  - Large gaps stay NaN → dropped later
  - WHY: Never fabricate target data for long periods

Temperature & Humidity:
  - Interpolate ≤12 hours — these change slowly
  
Wind Speed:
  - Interpolate ≤12 hours

Wind Direction:
  - Forward fill (ffill) ONLY — NOT interpolation
  - WHY: Direction is angular. Interpolating 350°→10° gives 180° (completely wrong!)
  - e.g., 270, NaN, NaN, 280 → 270, 270, 270, 280

Rainfall:
  - Create binary flag 'rainfall_missing' FIRST
  - Then fill NaN with 0
  - WHY: Lets models distinguish real zero rain from sensor-offline


In [ ]:
print("Missing values BEFORE cleaning:")
for stn in sorted(df_aq["station"].unique()):
    s = df_aq[df_aq["station"] == stn]
    print(f"  {stn:15s}: pm25={s['pm25'].isna().sum():5d}, "
          f"temp={s['temp'].isna().sum():5d}, "
          f"humidity={s['humidity'].isna().sum():5d}, "
          f"wind_dir={s['wind_direction'].isna().sum():5d}")

# Process each station SEPARATELY to avoid cross-station interpolation
cleaned_frames = []
for station_name in sorted(df_aq["station"].unique()):
    stn = df_aq[df_aq["station"] == station_name].copy()
    stn = stn.sort_values("datetime").reset_index(drop=True)

    # PM2.5: interpolate small gaps only (≤3 hours)
    stn["pm25"] = stn["pm25"].interpolate(method="linear", limit=3)

    # Temperature: interpolate ≤12 hours
    stn["temp"] = stn["temp"].interpolate(method="linear", limit=12)

    # Humidity: interpolate ≤12 hours
    stn["humidity"] = stn["humidity"].interpolate(method="linear", limit=12)

    # Wind Speed: interpolate ≤12 hours
    stn["wind_speed"] = stn["wind_speed"].interpolate(method="linear", limit=12)

    # Wind Direction: forward fill only (NOT interpolation)
    stn["wind_direction"] = stn["wind_direction"].ffill()

    # Rainfall: create missing flag THEN fill with 0
    stn["rainfall_missing"] = stn["rainfall"].isna().astype(int)
    stn["rainfall"] = stn["rainfall"].fillna(0)

    cleaned_frames.append(stn)

df_aq = pd.concat(cleaned_frames, ignore_index=True)

print("\nMissing values AFTER cleaning:")
for stn in sorted(df_aq["station"].unique()):
    s = df_aq[df_aq["station"] == stn]
    print(f"  {stn:15s}: pm25={s['pm25'].isna().sum():5d}, "
          f"temp={s['temp'].isna().sum():5d}, "
          f"humidity={s['humidity'].isna().sum():5d}, "
          f"wind_dir={s['wind_direction'].isna().sum():5d}")


Clean satellite fire detections:
  1. Parse datetime properly
  2. Remove invalid coordinates (outside Australia bounding box)
  3. Remove zero/negative FRP — no actual fire energy = false positive
  4. Floor datetime to hour for efficient matching with hourly AQ data

WHY: Satellite hotspot data contains spurious detections.
Zero FRP means no fire energy was measured. Flooring to hour enables
fast grouping when matching fires to hourly air quality observations.


In [ ]:
df_hotspots["datetime"] = pd.to_datetime(df_hotspots["datetime"], format="mixed")

initial_count = len(df_hotspots)
df_hotspots = df_hotspots.dropna(subset=["latitude", "longitude", "datetime"])
df_hotspots = df_hotspots[df_hotspots["power"] > 0]  # Remove zero/negative FRP
df_hotspots = df_hotspots[
    (df_hotspots["latitude"].between(-45, -10)) &   # Australia bounding box
    (df_hotspots["longitude"].between(110, 155))
]

# Floor to hour for efficient matching
df_hotspots["hourly_dt"] = df_hotspots["datetime"].dt.floor("h")
df_hotspots = df_hotspots.sort_values("datetime").reset_index(drop=True)

print(f"Hotspots: {initial_count:,} → {len(df_hotspots):,} after cleaning")
print(f"Date range: {df_hotspots['datetime'].min()} → {df_hotspots['datetime'].max()}")
print(f"\nFRP statistics:")
print(df_hotspots["power"].describe().round(2))


Save cleaned data in Parquet format.

WHY Parquet over CSV:
  - 5–20× smaller file size
  - Much faster read/write
  - Preserves data types (datetime, float, int) without re-parsing
  - Industry standard for data science pipelines


In [ ]:
os.makedirs("data", exist_ok=True)
os.makedirs("data/feature_store", exist_ok=True)

# Ensure numeric types before saving
for col in ["pm25", "temp", "humidity", "wind_speed", "wind_direction", "rainfall"]:
    if col in df_aq.columns:
        df_aq[col] = pd.to_numeric(df_aq[col], errors="coerce")

if "rainfall_missing" in df_aq.columns:
    df_aq["rainfall_missing"] = df_aq["rainfall_missing"].astype(int)

df_aq.to_parquet("data/air_quality_clean.parquet", index=False)
df_hotspots.to_parquet("data/hotspots_clean.parquet", index=False)
df_stations.to_csv("data/stations.csv", index=False)

print(f"✅ Saved air_quality_clean.parquet: {df_aq.shape}")
print(f"✅ Saved hotspots_clean.parquet: {df_hotspots.shape}")
print(f"✅ Saved stations.csv: {df_stations.shape}")


Vectorized geographic calculations for fire-exposure features.

Three functions:
  1. haversine_vectorized — Fast distance from station to ALL fires (numpy)
  2. calculate_bearing_vectorized — Compass bearing FROM station TO each fire
  3. compute_fire_features_for_window — ALL spatial features for one station-hour:
     - nearest_fire_km
     - fires_50km, fires_100km, fires_200km, fires_500km (counts within radii)
     - frp_50km, frp_100km, frp_200km, frp_500km (FRP sums within radii)
     - fire_exposure = Σ(FRP / distance²) — inverse square law
     - upwind_exposure = Σ(FRP × cos(wind_angle_diff) / distance²)

WHY upwind_exposure is the most physically meaningful fire variable:
  It combines three physical factors:
    INTENSITY (FRP) — larger fires produce more smoke
    DISTANCE (1/d²) — smoke disperses with distance squared
    WIND DIRECTION — fires UPWIND affect you, fires downwind don't
  A 100MW fire upwind at 50km is far more dangerous than a 100MW fire
  downwind at 10km. Simple fire count ignores all of this physics.


In [ ]:
def haversine_vectorized(lat1, lon1, lat2_arr, lon2_arr):
    """
    Vectorized Haversine distance calculation.
    lat1, lon1: scalar (station coordinates)
    lat2_arr, lon2_arr: numpy arrays (fire coordinates)
    Returns: distances in km
    """
    R = 6371.0  # Earth's radius in km
    lat1_r = np.radians(lat1)
    lat2_r = np.radians(lat2_arr)
    dlat = np.radians(lat2_arr - lat1)
    dlon = np.radians(lon2_arr - lon1)
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1_r) * np.cos(lat2_r) * np.sin(dlon / 2) ** 2
    c = 2 * np.arcsin(np.sqrt(np.clip(a, 0, 1)))
    return R * c


def calculate_bearing_vectorized(lat1, lon1, lat2_arr, lon2_arr):
    """
    Vectorized bearing FROM station TO fires.
    Returns bearings in degrees (0–360).
    """
    lat1_r = np.radians(lat1)
    lat2_r = np.radians(lat2_arr)
    dlon = np.radians(lon2_arr - lon1)
    x = np.sin(dlon) * np.cos(lat2_r)
    y = np.cos(lat1_r) * np.sin(lat2_r) - np.sin(lat1_r) * np.cos(lat2_r) * np.cos(dlon)
    bearing = np.degrees(np.arctan2(x, y))
    return (bearing + 360) % 360


def compute_fire_features_for_window(fire_lats, fire_lons, fire_powers,
                                      st_lat, st_lon, wind_dir):
    """
    Compute all spatial fire features for a given set of fires
    relative to one monitoring station at one time point.
    """
    result = {
        "nearest_fire_km": 999.0,
        "fires_50km": 0, "fires_100km": 0, "fires_200km": 0, "fires_500km": 0,
        "frp_50km": 0.0, "frp_100km": 0.0, "frp_200km": 0.0, "frp_500km": 0.0,
        "fire_exposure": 0.0,
        "upwind_exposure": 0.0,
    }

    if len(fire_lats) == 0:
        return result

    # Vectorized distance and bearing
    distances = haversine_vectorized(st_lat, st_lon, fire_lats, fire_lons)
    bearings = calculate_bearing_vectorized(st_lat, st_lon, fire_lats, fire_lons)
    powers = fire_powers

    # Nearest fire
    result["nearest_fire_km"] = float(np.min(distances))

    # Counts within radii
    for radius, key in [(50, "fires_50km"), (100, "fires_100km"),
                         (200, "fires_200km"), (500, "fires_500km")]:
        result[key] = int(np.sum(distances <= radius))

    # FRP sums within radii
    for radius, key in [(50, "frp_50km"), (100, "frp_100km"),
                         (200, "frp_200km"), (500, "frp_500km")]:
        result[key] = float(np.sum(powers[distances <= radius]))

    # Fire Exposure Index = Σ(FRP / distance²)
    valid = distances > 0
    if np.any(valid):
        result["fire_exposure"] = float(np.sum(powers[valid] / (distances[valid] ** 2)))

    # Upwind Exposure Index = Σ(FRP × cos(angle_diff) / distance²)
    if not np.isnan(wind_dir) and np.any(valid):
        angle_diff = np.abs(bearings[valid] - wind_dir)
        angle_diff = np.minimum(angle_diff, 360 - angle_diff)
        alignment = np.maximum(0, np.cos(np.radians(angle_diff)))
        result["upwind_exposure"] = float(
            np.sum(powers[valid] * alignment / (distances[valid] ** 2))
        )

    return result


# Pre-index hotspots by date for fast window lookups
df_hotspots["date"] = df_hotspots["datetime"].dt.date
hotspot_date_index = df_hotspots.groupby("date").apply(
    lambda g: g[["latitude", "longitude", "power", "datetime"]].values
).to_dict()


def get_fires_in_window(t_start, t_end):
    """
    Efficiently retrieve all fires in [t_start, t_end] time window.
    Uses date-based pre-indexing for speed.
    """
    start_date = t_start.date()
    end_date = t_end.date()

    fires_list = []
    current_date = start_date
    while current_date <= end_date:
        if current_date in hotspot_date_index:
            fires_list.append(hotspot_date_index[current_date])
        current_date += pd.Timedelta(days=1)

    if not fires_list:
        return np.array([]), np.array([]), np.array([])

    all_fires = np.vstack(fires_list)
    fire_datetimes = pd.to_datetime(all_fires[:, 3])
    mask = (fire_datetimes >= t_start) & (fire_datetimes <= t_end)

    if not np.any(mask):
        return np.array([]), np.array([]), np.array([])

    filtered = all_fires[mask]
    return (
        filtered[:, 0].astype(float),  # latitudes
        filtered[:, 1].astype(float),  # longitudes
        filtered[:, 2].astype(float),  # powers (FRP)
    )


print(f"✅ Spatial functions ready. Hotspot date index: {len(hotspot_date_index)} dates.")


Main fire feature computation loop.

For EVERY station-hour observation, we look at fires within a 24-HOUR
LOOKBACK WINDOW and compute spatial fire features.

WHY 24-HOUR LOOKBACK:
  Smoke from fires doesn't arrive instantly. A fire burning 200km away
  12 hours ago can still affect current air quality through atmospheric
  transport. Using only same-hour fires would miss the majority of
  smoke impacts. 24 hours captures the typical transport timescale
  for distances up to ~500km in typical Australian wind conditions.

PROCESS per station:
  1. Get station coordinates from Stations.csv
  2. For each hourly observation:
     a. Define window [t-24h, t]
     b. Retrieve all fires in that window
     c. Compute distances, bearings, FRP-weighted features
     d. Store results
  3. Save per-station Parquet file

RUNTIME: ~2–5 minutes per station depending on data size.


In [ ]:

LOOKBACK_HOURS = 24

station_list = sorted(df_aq["station"].unique())
print(f"Processing {len(station_list)} stations: {station_list}\n")

for station_name in station_list:

    print(f"{'=' * 60}")
    print(f"PROCESSING: {station_name}")
    print(f"{'=' * 60}")

    # Get station coordinates
    stn_row = df_stations[df_stations["station"].str.lower() == station_name.lower()]
    if len(stn_row) == 0:
        print(f"  ⚠️ No coordinates found for {station_name}, skipping!")
        continue

    st_lat = float(stn_row["latitude"].iloc[0])
    st_lon = float(stn_row["longitude"].iloc[0])

    # Get this station's air quality data
    stn_aq = df_aq[df_aq["station"] == station_name].copy()
    stn_aq = stn_aq.sort_values("datetime").reset_index(drop=True)
    print(f"  Coordinates: ({st_lat:.4f}, {st_lon:.4f})")
    print(f"  AQ records: {len(stn_aq):,}")
    print(f"  Date range: {stn_aq['datetime'].min()} → {stn_aq['datetime'].max()}")

    # Compute fire features for each hourly observation
    fire_feature_list = []
    start_time = time.time()

    for idx, row in stn_aq.iterrows():
        t_end = row["datetime"]
        t_start = t_end - pd.Timedelta(hours=LOOKBACK_HOURS)
        wind_dir = row.get("wind_direction", np.nan)
        if pd.isna(wind_dir):
            wind_dir = np.nan

        # Get fires in 24-hour lookback window
        f_lats, f_lons, f_powers = get_fires_in_window(t_start, t_end)

        # Compute all spatial fire features
        features = compute_fire_features_for_window(
            f_lats, f_lons, f_powers, st_lat, st_lon, wind_dir
        )
        fire_feature_list.append(features)

        # Progress reporting
        if (idx + 1) % 5000 == 0:
            elapsed = time.time() - start_time
            rate = (idx + 1) / elapsed
            remaining = (len(stn_aq) - idx - 1) / rate / 60
            print(f"    Row {idx+1:,}/{len(stn_aq):,} | "
                  f"{rate:.0f} rows/sec | ~{remaining:.1f} min remaining")

    # Convert fire features to DataFrame and merge
    df_fire = pd.DataFrame(fire_feature_list)
    stn_aq = pd.concat([stn_aq.reset_index(drop=True), df_fire], axis=1)

    # Save to feature store
    output_path = f"data/feature_store/{station_name}_with_fire.parquet"
    stn_aq.to_parquet(output_path, index=False)

    elapsed_total = (time.time() - start_time) / 60
    print(f"  ✓ Completed in {elapsed_total:.1f} minutes")
    print(f"  ✓ Saved: {output_path}\n")

print("=" * 60)
print("✅ ALL STATIONS FIRE FEATURES COMPUTED!")
print("=" * 60)


Load each station's fire-feature file and add ALL remaining features:

PHASE 4 — Temporal Fire Accumulations:
  frp_sum_6h/12h/24h/48h: Rolling sum of FRP over different windows
  fires_sum_24h: Rolling fire count in 24h
  WHY: Smoke accumulates. 48 hours of continuous burning produces
  worse air quality than a single-hour fire at the same intensity.

PHASE 5 — PM2.5 Memory Features (STRONGEST PREDICTORS):
  Lags: pm25_lag1/3/6/12/24 — PM2.5 is highly autocorrelated
  Rolling means: pm25_mean_6h/12h/24h — captures persistent episodes
  Rolling max: pm25_max_24h — captures worst recent conditions
  FRP lags: frp_lag1/3/6/12/24 — temporal memory of fire activity
  WHY: Current PM2.5 is the best predictor of next-hour PM2.5.
  These features capture "smoke memory."

PHASE 6 — Weather Features:
  wind_u, wind_v — decomposition from speed + direction
  WHY: Wind direction is circular (350° ≈ 10°). Decomposing into
  U/V components makes it linear and ML-friendly.

PHASE 7 — Seasonal Features:
  hour, month, Australian season (Summer = Dec–Feb)
  sin/cos cyclic encoding for hour and month
  WHY: Hour 23 is close to hour 0 numerically they're far apart.
  Sine/cosine encoding preserves this cyclical relationship.

PHASE 8 — Target Variable:
  target_pm25 = PM2.5 at (t+1) — the NEXT hour's PM2.5
  WHY: We predict FUTURE air quality. Using current PM2.5 as both
  feature and target = data leakage.


In [ ]:
def get_australian_season(month):
    """Returns Australian season name from month number."""
    if month in [12, 1, 2]:
        return "Summer"
    elif month in [3, 4, 5]:
        return "Autumn"
    elif month in [6, 7, 8]:
        return "Winter"
    else:
        return "Spring"


def engineer_all_features(df):
    """
    Apply all feature engineering phases to a single station DataFrame.
    Input must already have fire spatial features computed.
    Returns the fully engineered DataFrame.
    """
    df = df.sort_values("datetime").reset_index(drop=True)

    # ---- PHASE 4: Temporal Fire Accumulations ----
    df["frp_sum_6h"] = df["frp_500km"].rolling(6, min_periods=1).sum()
    df["frp_sum_12h"] = df["frp_500km"].rolling(12, min_periods=1).sum()
    df["frp_sum_24h"] = df["frp_500km"].rolling(24, min_periods=1).sum()
    df["frp_sum_48h"] = df["frp_500km"].rolling(48, min_periods=1).sum()
    df["fires_sum_24h"] = df["fires_500km"].rolling(24, min_periods=1).sum()

    # ---- PHASE 5: PM2.5 Memory + FRP Lags ----
    for lag in [1, 3, 6, 12, 24]:
        df[f"pm25_lag{lag}"] = df["pm25"].shift(lag)
        df[f"frp_lag{lag}"] = df["frp_500km"].shift(lag)

    df["pm25_mean_6h"] = df["pm25"].rolling(6, min_periods=1).mean()
    df["pm25_mean_12h"] = df["pm25"].rolling(12, min_periods=1).mean()
    df["pm25_mean_24h"] = df["pm25"].rolling(24, min_periods=1).mean()
    df["pm25_max_24h"] = df["pm25"].rolling(24, min_periods=1).max()

    # ---- PHASE 6: Weather Features ----
    wind_dir_rad = np.radians(df["wind_direction"].fillna(0))
    wind_spd = df["wind_speed"].fillna(0)
    df["wind_u"] = wind_spd * np.cos(wind_dir_rad)
    df["wind_v"] = wind_spd * np.sin(wind_dir_rad)

    # ---- PHASE 7: Seasonal Features ----
    df["hour"] = df["datetime"].dt.hour
    df["month"] = df["datetime"].dt.month
    df["season"] = df["month"].apply(get_australian_season)
    df["sin_hour"] = np.sin(2 * np.pi * df["hour"] / 24.0)
    df["cos_hour"] = np.cos(2 * np.pi * df["hour"] / 24.0)
    df["sin_month"] = np.sin(2 * np.pi * df["month"] / 12.0)
    df["cos_month"] = np.cos(2 * np.pi * df["month"] / 12.0)

    # ---- PHASE 8: Target Variable ----
    # target_pm25 = PM2.5 at (t+1) — prevents data leakage
    df["target_pm25"] = df["pm25"].shift(-1)

    return df


# Process each station
for station_name in station_list:
    input_path = f"data/feature_store/{station_name}_with_fire.parquet"

    if not os.path.exists(input_path):
        print(f"  Skipping {station_name} — fire features not found")
        continue

    print(f"Engineering features for {station_name}...")

    df_stn = pd.read_parquet(input_path)
    df_stn = engineer_all_features(df_stn)

    # Drop rows where critical features are NaN (from lag/rolling calculations)
    df_stn_clean = df_stn.dropna(subset=["pm25_lag24", "target_pm25"]).reset_index(drop=True)

    output_path = f"data/feature_store/{station_name}_features.parquet"
    df_stn_clean.to_parquet(output_path, index=False)

    print(f"  ✓ {station_name}: {len(df_stn):,} → {len(df_stn_clean):,} rows | "
          f"{len(df_stn_clean.columns)} features")

print("\n✅ All stations fully feature-engineered!")


Combine all station Parquet files into one dataset.
Define the shared feature list used by all three students.
Create temporal train/test split.

TEMPORAL SPLIT:
  Train: 2016–2018 (pre-Black Summer)
  Test:  2019–2020 (includes Black Summer — hardest possible test)

WHY NOT RANDOM SPLIT:
  Air quality observations close in time are highly correlated.
  Random split leaks future information into training, giving
  unrealistically good results. Temporal split simulates real
  forecasting: can models trained on "normal" years predict the
  worst fire season in Australian history?


In [ ]:
# Define the shared feature columns
FEATURE_COLS = [
    # Weather
    "temp", "humidity", "rainfall", "rainfall_missing",
    "wind_speed", "wind_u", "wind_v",
    # Fire spatial
    "nearest_fire_km",
    "fires_50km", "fires_100km", "fires_200km", "fires_500km",
    "frp_50km", "frp_100km", "frp_200km", "frp_500km",
    "fire_exposure", "upwind_exposure",
    # Fire temporal accumulations
    "frp_sum_6h", "frp_sum_12h", "frp_sum_24h", "frp_sum_48h",
    "fires_sum_24h",
    # FRP lags
    "frp_lag1", "frp_lag3", "frp_lag6", "frp_lag12", "frp_lag24",
    # PM2.5 memory
    "pm25_lag1", "pm25_lag3", "pm25_lag6", "pm25_lag12", "pm25_lag24",
    "pm25_mean_6h", "pm25_mean_12h", "pm25_mean_24h",
    "pm25_max_24h",
    # Time / seasonal
    "hour", "month",
    "sin_hour", "cos_hour", "sin_month", "cos_month",
]

TARGET_COL = "target_pm25"

# Load and combine all stations
all_stations = []
for station_name in station_list:
    fpath = f"data/feature_store/{station_name}_features.parquet"
    if os.path.exists(fpath):
        df_stn = pd.read_parquet(fpath)
        all_stations.append(df_stn)
        print(f"  Loaded {station_name}: {len(df_stn):,} rows")
    else:
        print(f"  ⚠️ {station_name}: features file not found")

df_all = pd.concat(all_stations, ignore_index=True)

# Encode station as numeric ID
station_encoder = LabelEncoder()
df_all["station_id"] = station_encoder.fit_transform(df_all["station"])

FEATURE_COLS_WITH_STATION = FEATURE_COLS + ["station_id"]

# Temporal split
TRAIN_END = "2018-12-31"
TEST_START = "2019-01-01"
TEST_END = "2020-12-31"

df_train = df_all[df_all["datetime"] <= TRAIN_END].dropna(
    subset=FEATURE_COLS_WITH_STATION + [TARGET_COL]
)
df_test = df_all[
    (df_all["datetime"] >= TEST_START) & (df_all["datetime"] <= TEST_END)
].dropna(subset=FEATURE_COLS_WITH_STATION + [TARGET_COL])

X_train = df_train[FEATURE_COLS_WITH_STATION]
y_train = df_train[TARGET_COL]
X_test = df_test[FEATURE_COLS_WITH_STATION]
y_test = df_test[TARGET_COL]

print(f"\n{'=' * 60}")
print(f"Combined dataset: {df_all.shape}")
print(f"Features: {len(FEATURE_COLS_WITH_STATION)}")
print(f"{'=' * 60}")
print(f"Train: {len(X_train):,} samples "
      f"({df_train['datetime'].min()} → {df_train['datetime'].max()})")
print(f"Test:  {len(X_test):,} samples "
      f"({df_test['datetime'].min()} → {df_test['datetime'].max()})")
print(f"\nTrain stations: {df_train['station'].value_counts().to_dict()}")
print(f"Test stations:  {df_test['station'].value_counts().to_dict()}")


Shared EDA visualisations:
  1. PM2.5 distribution by station (boxplot)
  2. Monthly PM2.5 pattern (fire season = Oct–Mar in Australia)
  3. PM2.5 vs FRP 500km (fire intensity relationship)
  4. PM2.5 vs Nearest Fire Distance
  5. PM2.5 vs Upwind Exposure Index
  6. Yearly mean PM2.5 (Black Summer spike visible)
  7. Feature correlation heatmap
  8. Top 15 features correlated with target


In [ ]:
# ---- 6-Panel EDA ----
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# 1. PM2.5 by station
ax = axes[0, 0]
df_all.boxplot(column="target_pm25", by="station", ax=ax, showfliers=False)
ax.set_title("PM2.5 Distribution by Station")
ax.set_xlabel("Station"); ax.set_ylabel("PM2.5 (µg/m³)")
plt.sca(ax); plt.xticks(rotation=45, ha="right")

# 2. Monthly pattern
ax = axes[0, 1]
monthly = df_all.groupby("month")["target_pm25"].mean()
ax.bar(monthly.index, monthly.values, color="coral")
ax.set_title("Mean PM2.5 by Month"); ax.set_xlabel("Month")
ax.set_ylabel("Mean PM2.5"); ax.set_xticks(range(1, 13))

# 3. PM2.5 vs FRP
ax = axes[0, 2]
ax.scatter(df_all["frp_500km"], df_all["target_pm25"], alpha=0.1, s=1, c="red")
ax.set_title("PM2.5 vs FRP (500km)")
ax.set_xlabel("Total FRP within 500km"); ax.set_ylabel("PM2.5")
ax.set_xlim(0, df_all["frp_500km"].quantile(0.99))

# 4. PM2.5 vs Nearest Fire
ax = axes[1, 0]
ax.scatter(df_all["nearest_fire_km"], df_all["target_pm25"], alpha=0.1, s=1, c="blue")
ax.set_title("PM2.5 vs Nearest Fire Distance")
ax.set_xlabel("Nearest Fire (km)"); ax.set_ylabel("PM2.5")

# 5. PM2.5 vs Upwind Exposure
ax = axes[1, 1]
ax.scatter(df_all["upwind_exposure"], df_all["target_pm25"], alpha=0.1, s=1, c="green")
ax.set_title("PM2.5 vs Upwind Exposure Index")
ax.set_xlabel("Upwind Exposure"); ax.set_ylabel("PM2.5")
ax.set_xlim(0, df_all["upwind_exposure"].quantile(0.99))

# 6. Yearly mean
ax = axes[1, 2]
df_all["year"] = df_all["datetime"].dt.year
yearly = df_all.groupby("year")["target_pm25"].mean()
ax.bar(yearly.index, yearly.values, color="darkorange")
ax.set_title("Mean PM2.5 by Year (Black Summer = 2019–2020)")
ax.set_xlabel("Year"); ax.set_ylabel("Mean PM2.5")
ax.axhline(y=yearly.mean(), color="gray", linestyle="--", alpha=0.5, label="Overall Mean")
ax.legend()

plt.suptitle("Exploratory Data Analysis — All Stations", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig("EDA_all_stations.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ EDA plot saved as EDA_all_stations.png")

# ---- Correlation Heatmap ----
corr_cols = [c for c in FEATURE_COLS + ["target_pm25"] if c in df_all.columns]
corr_data = df_all[corr_cols].dropna()
corr_matrix = corr_data.corr()

plt.figure(figsize=(20, 16))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=False, cmap="RdBu_r", center=0,
            vmin=-1, vmax=1, square=True, linewidths=0.5)
plt.title("Feature Correlation Matrix — All Features", fontsize=14)
plt.tight_layout()
plt.savefig("EDA_correlation_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Correlation heatmap saved")

# Top correlations with target
target_corr = corr_matrix["target_pm25"].drop("target_pm25").abs().sort_values(ascending=False)
print("\nTop 15 features correlated with target PM2.5:")
for feat, corr_val in target_corr.head(15).items():
    sign = "+" if corr_matrix.loc[feat, "target_pm25"] > 0 else "-"
    print(f"  {feat:25s}: {sign}{corr_val:.4f}")